# Entrenamiento y Optimización de hiperparametros

Señales vistas en el EDA:
- Tamaño del dataset: TEnemos un dataset con pocos datos (5.1k) lo cual favorece a modelos simples e interpretables (KNN)
- Número de features: por lo que también favorece modelos de menor complejidad.
- Dataset altamente desbalanceado: preferimos modelos que soporten nativamente `class_weights`. Decidimos no usar oversampling o undersampling, con tan pocos positivos la interpolación generaría vecinos sintéticos muy cercanos a los originales, lo que podría inflar las métricas. De todos modos, se hizo un SMOTE-NC y no generó mejora de performance.
- Necesidad de intrepretabilidad, como arboles simples


### Baseline
- Dummy Classifier como baseline: El dataset tiene un fuerte desbalance, por lo que un modelo que prediga siempre la clase mayoritaria (no-stroke) ya tiene un 95% de precisión. Referencia para justificar ML.

### Elegimos los siguientes modelos:

- KNN: Un modelo basado en distancia podria funcionar en nuestro dataset ya que es lo suficientemente pequeño para que sea viable computacionalmente. Permite explorar si la similitud entre pacientes en el espacio de features (edad, glucosa, bmi) es suficiente para distinguir casos de stroke. Sin embargo, un fuerte desbalance afectará el performance del modelo.

- Decision Tree (Arbol simple): el riesgo de stroke tiene relaciones no lineales con features como la edad, glucosa y bmi, un arbol puede capturar con particiones simples estos casos. Además, ofrece un buen grado de interpretabilidad.

- Random Forest (Ensamble bagging): Es esperado que un solo arbol tiende a overfit, algo que random forest mitigaría.

- XGBoost (Ensamble Boosting): A diferencia de Random Forest, XGBoost corrige errores iterativamente lo que podria ser el modelo mas potente de esta seleccion, por lo que seria interesante ver su desempeño con el resto de los modelos.
  
- SVC: busca el hiperplano de maximo margen entre clases. Al usar un kernel polinomial o radial puede capturar fronteras de decision no lineales, lo que lo hace relevante segun lo visto en el EDA.

Contexto **médico** (predicción de stroke) con clases altamente desbalanceadas (~19.5:1). Un **falso negativo**, no detectar un stroke para alguien que si esta en riesgo es un error catastrófico, mientras que un falso positivo, aunque indeseable, tiene consecuencias mucho menores.

Usamos **F-beta score** (β=2): Recall pesa ~4× más que Precision, priorizando que el modelo no se pierda casos reales de stroke. A diferencia del **recall** puro, el f-beta sigue penalizando en modelos que predicen 'stroke' para todos, manteniendo un grado de robustez en precision.

Para ajustar el peso de Recall, cambiar `BETA` en la celda de setup.

In [12]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.base import BaseEstimator, ClassifierMixin

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.metrics import (
    make_scorer, fbeta_score,
    roc_auc_score, average_precision_score,
    f1_score, recall_score, precision_score,
    confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay,
)

import warnings
warnings.filterwarnings('ignore')

# El nb01 genera 3 splits: train / val / test.
#  - X_train: CV 5-fold para tuning de hiperparámetros.
#  - X_val:   comparación de los modelos tuneados para elegir al ganador.
#  - X_test:  evaluación final del modelo ganador (holdout intocado).
# Nota: en el pkl las claves están invertidas ('X_test' → val, 'X_val' → test).
data = joblib.load('../data/processed_data.pkl')
X_train = data['X_train']
X_val   = data['X_test']   # pkl key 'X_test' es el set de validación (selección de modelo)
X_test  = data['X_val']    # pkl key 'X_val' es el holdout final
y_train = data['y_train']
y_val   = data['y_test']
y_test  = data['y_val']

print(f'X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')
print(f'Distribución de clases (train): {np.bincount(y_train.astype(int))}')
print(f'Ratio de desbalance: {np.sum(y_train==0)/np.sum(y_train==1):.1f}:1')

X_train: (3065, 14) | X_val: (1022, 14) | X_test: (1022, 14)
Distribución de clases (train): [2916  149]
Ratio de desbalance: 19.6:1


## Setup 

En esta celda se setea: el scorer F-beta, la CV estratificada 5-fold, `scale_pos_weight` para XGBoost, y un helper `cv_f2` que evalúa un modelo con CV=5 y devuelve el F-beta medio (lo que cada `objective` de Optuna maximiza).

Tenemos una validacion cruzada de 5 folds

**Por qué Optuna?:** Optuna con `TPESampler` (Tree-structured Parzen Estimator) hace búsqueda bayesiana — aprende de los trials previos y converge más rápido que random search o gridsearch.

In [13]:
BETA = 2  # para podrr ajustar el f-score. Hay que reentrenar cada vez que se cambia
METRIC_NAME = f'F{BETA}'  # se usa como clave de métricas en dicts/tablas
f2_scorer = make_scorer(fbeta_score, beta=BETA)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# scale_pos_weight para XGBoost (no tiene `class_weight`)
neg, pos = np.bincount(y_train)
scale_pos = neg / pos

# Dict de métricas para reporting en la tabla comparativa
scoring = {
    METRIC_NAME: f2_scorer,
    'Recall':    'recall',
    'Precision': 'precision',
    'ROC_AUC':   'roc_auc',
    'PR_AUC':    'average_precision',
}

# Silenciar logs verbosos de Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Helper: evalúa un modelo con CV=5 y devuelve F-beta medio (lo que cada objective maximiza)
def cv_f2(model, X, y):
    return cross_val_score(model, X, y, cv=cv, scoring=f2_scorer, n_jobs=-1).mean()

# Helper: imprime todas las métricas relevantes del mejor modelo en CV
def print_cv_metrics(name, model):
    s = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    print(f'\nMétricas CV — {name}:')
    for metric, key in [
        (METRIC_NAME,  f'test_{METRIC_NAME}'),
        ('Recall',     'test_Recall'),
        ('Precision',  'test_Precision'),
        ('ROC-AUC',    'test_ROC_AUC'),
        ('PR-AUC',     'test_PR_AUC'),
    ]:
        print(f'  {metric:<12} {s[key].mean():.4f}')

## Dummy Classifier

Baseline de referencia: predice siempre la clase mayoritaria (`most_frequent`). No hay hiperparámetros que optimizar — se evalúa con CV para establecer el piso de métricas que cualquier modelo real debe superar.

In [14]:
dummy_best = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)

print('Parámetros (Dummy Classifier):')
print('  strategy: most_frequent')
print_cv_metrics('Dummy Classifier', dummy_best)

Parámetros (Dummy Classifier):
  strategy: most_frequent

Métricas CV — Dummy Classifier:
  F2           0.0000
  Recall       0.0000
  Precision    0.0000
  ROC-AUC      0.5000
  PR-AUC       0.0486


/Users/alejandrovalle/Desktop/Posgrado/CEIA/2do_bimestre/aprendizaje_de_maquina/ML/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/alejandrovalle/Desktop/Posgrado/CEIA/2do_bimestre/aprendizaje_de_maquina/ML/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/alejandrovalle/Desktop/Posgrado/CEIA/2do_bimestre/aprendizaje_de_maquina/ML/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0

In [15]:
class AgeThresholdClassifier(BaseEstimator, ClassifierMixin):
    """
    Clasificador baseline:
    predice clase 1 si age > threshold, caso contrario clase 0.
    """

    def __init__(self, feature_name="age", threshold=50):
        self.feature_name = feature_name
        self.threshold = threshold

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = list(X.columns)
        else:
            raise ValueError(
                "Para usar este clasificador, X debe ser un DataFrame con nombres de columnas."
            )

        if self.feature_name not in self.feature_names_in_:
            raise ValueError(f"La feature '{self.feature_name}' no está en X.")

        self.feature_idx_ = self.feature_names_in_.index(self.feature_name)
        self.classes_ = np.array([0, 1])

        return self

    def predict(self, X):
        if hasattr(X, "iloc"):
            age_values = X.iloc[:, self.feature_idx_].values
        else:
            age_values = X[:, self.feature_idx_]

        return (age_values > self.threshold).astype(int)

    def predict_proba(self, X):
        y_pred = self.predict(X)
        return np.column_stack([1 - y_pred, y_pred])


# X_train es numpy; el clasificador requiere DataFrame para obtener feature names en fit
X_train_df = pd.DataFrame(X_train, columns=data['feature_names'])

age_clf = AgeThresholdClassifier(feature_name="age", threshold=50)
age_clf.fit(X_train_df, y_train)

y_pred_age_val = age_clf.predict(X_val)
y_proba_age_val = age_clf.predict_proba(X_val)[:, 1]

print(f"\nAge Baseline (age > 50):")
print(f"  {METRIC_NAME:<12} {fbeta_score(y_val, y_pred_age_val, beta=BETA):.4f}")
print(f"  {'Recall':<12} {recall_score(y_val, y_pred_age_val):.4f}")
print(f"  {'Precision':<12} {precision_score(y_val, y_pred_age_val, zero_division=0):.4f}")
print(f"  {'ROC-AUC':<12} {roc_auc_score(y_val, y_proba_age_val):.4f}")
print(f"  {'PR-AUC':<12} {average_precision_score(y_val, y_proba_age_val):.4f}")


Age Baseline (age > 50):
  F2           0.0000
  Recall       0.0000
  Precision    0.0000
  ROC-AUC      0.5000
  PR-AUC       0.0489


Como podemos ver es un modelo que adivina. Otra alternativa podria ser un baseline que detecte stroke si la variable age > 50 (visto en el eda), que viene a ser lo mismo ya que la gran mayoria de stroke occurren cuando el paciente es mayor a 50 años

## KNN

Espacio de búsqueda: número de vecinos, tipo de peso y métrica (`p=1` Manhattan, `p=2` Euclidiana)

In [16]:
def knn_objective(trial):
    params = {
        'n_neighbors': trial.suggest_int('n_neighbors', 3, 30),
        'weights':     trial.suggest_categorical('weights', ['uniform', 'distance']),
        'p':           trial.suggest_categorical('p', [1, 2]),
    }
    model = KNeighborsClassifier(n_jobs=-1, **params)
    return cv_f2(model, X_train, y_train)

knn_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
knn_study.optimize(knn_objective, n_trials=50, show_progress_bar=True)

knn_best_params = knn_study.best_params
knn_best = KNeighborsClassifier(n_jobs=-1, **knn_best_params).fit(X_train, y_train)

print('Mejores parámetros (KNN):')
for k, v in knn_best_params.items():
    print(f'  {k}: {v}')
print_cv_metrics('KNN', knn_best)

  0%|          | 0/50 [00:00<?, ?it/s]

Mejores parámetros (KNN):
  n_neighbors: 4
  weights: distance
  p: 1

Métricas CV — KNN:
  F2           0.0773
  Recall       0.0674
  Precision    0.2000
  ROC-AUC      0.6463
  PR-AUC       0.1161


## Decision Tree

Espacio de búsqueda: profundidad, criterio de split y regularización (`min_samples_*`).

In [17]:
def dt_objective(trial):
    params = {
        'max_depth':         trial.suggest_int('max_depth', 3, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'criterion':         trial.suggest_categorical('criterion', ['gini', 'entropy']),
    }
    model = DecisionTreeClassifier(class_weight='balanced', random_state=42, **params)
    return cv_f2(model, X_train, y_train)

dt_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
dt_study.optimize(dt_objective, n_trials=50, show_progress_bar=True)

dt_best_params = dt_study.best_params
dt_best = DecisionTreeClassifier(
    class_weight='balanced', random_state=42, **dt_best_params
).fit(X_train, y_train)

print('Mejores parámetros (Decision Tree):')
for k, v in dt_best_params.items():
    print(f'  {k}: {v}')
print_cv_metrics('Decision Tree', dt_best)

  0%|          | 0/50 [00:00<?, ?it/s]

Mejores parámetros (Decision Tree):
  max_depth: 15
  min_samples_leaf: 43
  min_samples_split: 18
  criterion: gini

Métricas CV — Decision Tree:
  F2           0.4003
  Recall       0.8057
  Precision    0.1341
  ROC-AUC      0.7782
  PR-AUC       0.1381


## Random Forest

Espacio de búsqueda: número de árboles, profundidad, regularización y muestreo de features.

In [18]:
def rf_objective(trial):
    params = {
        'n_estimators':     trial.suggest_categorical('n_estimators', [100, 200, 300, 500]),
        'max_depth':        trial.suggest_categorical('max_depth', [5, 10, 15, 20, None]),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features':     trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
    }
    model = RandomForestClassifier(
        class_weight='balanced', random_state=42, n_jobs=-1, **params
    )
    return cv_f2(model, X_train, y_train)

rf_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
rf_study.optimize(rf_objective, n_trials=50, show_progress_bar=True)

rf_best_params = rf_study.best_params
rf_best = RandomForestClassifier(
    class_weight='balanced', random_state=42, n_jobs=-1, **rf_best_params
).fit(X_train, y_train)

print('Mejores parámetros (Random Forest):')
for k, v in rf_best_params.items():
    print(f'  {k}: {v}')
print_cv_metrics('Random Forest', rf_best)

  0%|          | 0/50 [00:00<?, ?it/s]

Mejores parámetros (Random Forest):
  n_estimators: 100
  max_depth: 10
  min_samples_leaf: 19
  max_features: log2

Métricas CV — Random Forest:
  F2           0.4092
  Recall       0.7182
  Precision    0.1505
  ROC-AUC      0.8426
  PR-AUC       0.1862


## XGBoost

Espacio de búsqueda más rico: learning rate, subsampling de filas y columnas, `min_child_weight`

In [19]:
def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_categorical('n_estimators', [100, 200, 300, 500]),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    model = XGBClassifier(
        scale_pos_weight=scale_pos,
        eval_metric='aucpr',
        random_state=42,
        use_label_encoder=False,
        n_jobs=1,
        **params,
    )
    return cv_f2(model, X_train, y_train)

xgb_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

xgb_best_params = xgb_study.best_params
xgb_best = XGBClassifier(
    scale_pos_weight=scale_pos,
    eval_metric='aucpr',
    random_state=42,
    use_label_encoder=False,
    n_jobs=1,
    **xgb_best_params,
).fit(X_train, y_train)

print('Mejores parámetros (XGBoost):')
for k, v in xgb_best_params.items():
    print(f'  {k}: {v}')
print_cv_metrics('XGBoost', xgb_best)

  0%|          | 0/50 [00:00<?, ?it/s]

Mejores parámetros (XGBoost):
  n_estimators: 200
  max_depth: 3
  learning_rate: 0.05508629584267081
  min_child_weight: 5
  subsample: 0.7143519737735834
  colsample_bytree: 0.8839578671825332

Métricas CV — XGBoost:
  F2           0.4166
  Recall       0.6986
  Precision    0.1597
  ROC-AUC      0.8389
  PR-AUC       0.1935


## SVC

Espacio de búsqueda con 4 kernels (linear/rbf/poly/sigmoid). Se usa `class_weight='balanced'` y `probability=True` para poder calcular AUC-ROC y PR-AUC. El parámetro `C` y `gamma` dependen del kernel (espacios condicionales).



In [20]:
def svc_objective(trial):
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])

    if kernel == 'linear':
        params = {
            'kernel': kernel,
            'C': trial.suggest_float('C_linear', 1e-3, 1e2, log=True),
        }
    elif kernel == 'rbf':
        params = {
            'kernel': kernel,
            'C':     trial.suggest_float('C_rbf', 1e-2, 1e2, log=True),
            'gamma': trial.suggest_categorical('gamma_rbf',
                ['scale', 'auto', 0.001, 0.005, 0.01, 0.05, 0.1]),
        }
    elif kernel == 'poly':
        params = {
            'kernel': kernel,
            'C':      trial.suggest_categorical('C_poly', [0.1, 1.0, 10.0]),
            'degree': trial.suggest_int('degree', 2, 4),
            'gamma':  trial.suggest_categorical('gamma_poly', ['scale', 'auto']),
        }
    else:  # sigmoid
        params = {
            'kernel': kernel,
            'C':     trial.suggest_categorical('C_sig', [0.1, 1.0, 10.0]),
            'gamma': trial.suggest_categorical('gamma_sig', ['scale', 'auto']),
        }

    model = SVC(
        class_weight='balanced',
        probability=True,
        random_state=42,
        **params,
    )
    return cv_f2(model, X_train, y_train)


def build_svc_from_params(best_params):
    kernel = best_params['kernel']
    build = dict(kernel=kernel, class_weight='balanced', probability=True, random_state=42)
    if kernel == 'linear':
        build['C'] = best_params['C_linear']
    elif kernel == 'rbf':
        build['C']     = best_params['C_rbf']
        build['gamma'] = best_params['gamma_rbf']
    elif kernel == 'poly':
        build['C']      = best_params['C_poly']
        build['degree'] = best_params['degree']
        build['gamma']  = best_params['gamma_poly']
    else:  # sigmoid
        build['C']     = best_params['C_sig']
        build['gamma'] = best_params['gamma_sig']
    return SVC(**build)


svc_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
svc_study.optimize(svc_objective, n_trials=50, show_progress_bar=True)

svc_best_params = svc_study.best_params
svc_best = build_svc_from_params(svc_best_params).fit(X_train, y_train)

print('Mejores parámetros (SVC):')
for k, v in svc_best_params.items():
    print(f'  {k}: {v}')
print_cv_metrics('SVC', svc_best)

  0%|          | 0/50 [00:00<?, ?it/s]

Mejores parámetros (SVC):
  kernel: poly
  C_poly: 0.1
  degree: 2
  gamma_poly: scale

Métricas CV — SVC:
  F2           0.4097
  Recall       0.7924
  Precision    0.1398
  ROC-AUC      0.8444
  PR-AUC       0.1999


## Resumen de Cross-Validation

Tabla con las métricas CV de la mejor configuración de cada modelo. La columna F-beta es la que se usó para seleccionar.

In [21]:
def cv_row(name, model):
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    return {
        'Modelo':    name,
        METRIC_NAME: scores[f'test_{METRIC_NAME}'].mean(),
        'Recall':    scores['test_Recall'].mean(),
        'Precision': scores['test_Precision'].mean(),
        'ROC-AUC':   scores['test_ROC_AUC'].mean(),
        'PR-AUC':    scores['test_PR_AUC'].mean(),
    }

cv_summary = (
    pd.DataFrame([
        cv_row('Dummy Classifier', dummy_best),
        cv_row('KNN',              knn_best),
        cv_row('Decision Tree',    dt_best),
        cv_row('Random Forest',    rf_best),
        cv_row('XGBoost',          xgb_best),
        cv_row('SVC',              svc_best),
    ])
    .set_index('Modelo')
    .sort_values(METRIC_NAME, ascending=False)
)
display(cv_summary.style.highlight_max(axis=0).format('{:.4f}'))

/Users/alejandrovalle/Desktop/Posgrado/CEIA/2do_bimestre/aprendizaje_de_maquina/ML/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/alejandrovalle/Desktop/Posgrado/CEIA/2do_bimestre/aprendizaje_de_maquina/ML/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/alejandrovalle/Desktop/Posgrado/CEIA/2do_bimestre/aprendizaje_de_maquina/ML/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0

,F2,Recall,Precision,ROC-AUC,PR-AUC
Modelo,,,,,
XGBoost,0.4166,0.6986,0.1597,0.8389,0.1935
SVC,0.4097,0.7924,0.1398,0.8444,0.1999
Random Forest,0.4092,0.7182,0.1505,0.8426,0.1862
Decision Tree,0.4003,0.8057,0.1341,0.7782,0.1381
KNN,0.0773,0.0674,0.2000,0.6463,0.1161
Dummy Classifier,0.0000,0.0000,0.0000,0.5000,0.0486


Como era de esperarse, la precisión resulta muy baja en todos los modelos. Tenemos un desbalance extremo y además que estamos priorizando recall

In [22]:
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(dummy_best, '../models/dummy_classifier.pkl')
joblib.dump(knn_best,   '../models/knn_tuned.pkl')
joblib.dump(dt_best,    '../models/decision_tree_tuned.pkl')
joblib.dump(rf_best,    '../models/random_forest_tuned.pkl')
joblib.dump(xgb_best,   '../models/xgboost_tuned.pkl')
joblib.dump(svc_best,   '../models/svc_tuned.pkl')
print('Modelos guardados en ../models/')

feature_names = list(X_train.columns) if hasattr(X_train, 'columns') else None

joblib.dump({
    'best_params': {
        'Dummy Classifier': {'strategy': 'most_frequent'},
        'KNN':              knn_study.best_params,
        'Decision Tree':    dt_study.best_params,
        'Random Forest':    rf_study.best_params,
        'XGBoost':          xgb_study.best_params,
        'SVC':              svc_study.best_params,
    },
    'cv_best_fbeta': {
        'Dummy Classifier': None,
        'KNN':              knn_study.best_value,
        'Decision Tree':    dt_study.best_value,
        'Random Forest':    rf_study.best_value,
        'XGBoost':          xgb_study.best_value,
        'SVC':              svc_study.best_value,
    },
    'beta':          BETA,
    'metric_name':   METRIC_NAME,
    'feature_names': feature_names,
}, '../data/tuning_results.pkl')
print('Parámetros de tuning guardados en ../data/tuning_results.pkl')

Modelos guardados en ../models/
Parámetros de tuning guardados en ../data/tuning_results.pkl


### Conclusiones
- Se realizó el entrenamiento de los modelos seleccionados y búsqueda de hiperpárametros utilizando optuna priorizando F-beta (2).

- Se intentó priorizar otras métricas como `recall`, f-beta con valores hasta beta 4, pero los modelos perdian bastante precision. Se predecian pocos falsos negativos pero muchos falsos positivos. F-beta con beta = 2 fue el mejor balance encontrado.

- En la notebook `03_model_comparison` se realizan las comparaciones de los modelos.